# 050 Blend Add047 + 048 + 049 Check

Blend/correlation check after adding 047, 048, and 049 to the 046 candidate pool.

Purpose:
- Test whether 049, which improved Public LB over 045/046, can improve direct probability blends with 040/038/033/045.
- Include 048 as a stronger OVR XGB variant material.
- Include 047 only as a low-priority diagnostic; if 047 receives non-trivial safe-blend weight, consider a later GPU LogReg stacker add047 check.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260614_050_blend_add047_048_049_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key": "015_realmlp_seed42", "exp_id": "exp_20260605_015_shared001_realmlp_as_is", "family": "RealMLP", "role": "stable_single_realmlp_backup", "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "cv": 0.9681693449222352, "public_lb": 0.96977},
    {"key": "017_realmlp_bias", "exp_id": "exp_20260606_017_bias_search_on_015_realmlp", "family": "RealMLP", "role": "previous_best_realmlp_bias_backup", "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "cv": 0.9683022653955233, "public_lb": 0.96985},
    {"key": "018_xgb_shared003", "exp_id": "exp_20260606_018_shared003_xgb_as_is", "family": "XGBoost", "role": "effective_blend_material", "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy", "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy", "cv": 0.965207986418628, "public_lb": 0.96578},
    {"key": "019_blend_best", "exp_id": "exp_20260607_019_blend_add018_xgb_check", "family": "Blend", "role": "previous_public_confirmed_best", "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "cv": 0.968872315017554, "public_lb": 0.97000},
    {"key": "020_blend_bias", "exp_id": "exp_20260607_020_bias_search_on_019_best_blend", "family": "Blend", "role": "cv_trusted_attack_reference", "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "cv": 0.9692240443617589, "public_lb": 0.96969},
    {"key": "024_seed_ensemble_blend", "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check", "family": "Blend", "role": "seed_ensemble_blend_reference", "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "cv": 0.9694803109983217, "public_lb": 0.96958},
    {"key": "026_realmlp_v5", "exp_id": "exp_20260608_026_realmlp_v5_as_is", "family": "RealMLP", "role": "realmlp_v5_single_model_candidate", "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy", "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy", "cv": 0.9690389777981231, "public_lb": 0.96979},
    {"key": "027_blend_add026", "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check", "family": "Blend", "role": "previous_cv_trusted_slot", "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "cv": 0.9695425457477898, "public_lb": 0.96975},
    {"key": "028_cat_v3", "exp_id": "exp_20260608_028_cat_v3_as_is", "family": "CatBoost", "role": "catboost_v3_family_aux_material", "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy", "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy", "cv": 0.9688146470512534, "public_lb": 0.96969},
    {"key": "029_blend_add028", "exp_id": "exp_20260608_029_blend_add028_cat_v3_check", "family": "Blend", "role": "previous_best_backup", "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "cv": 0.9700023026377228, "public_lb": 0.970036},
    {"key": "030_ovr_xgb", "exp_id": "exp_20260608_030_ovr_xgb_as_is", "family": "XGBoost", "role": "low_weight_auxiliary_xgb_ovr_material", "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy", "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy", "cv": 0.9609971296999271, "public_lb": 0.96118},
    {"key": "031_blend_add030", "exp_id": "exp_20260608_031_blend_add030_ovr_xgb_check", "family": "Blend", "role": "public_confirmed_current_best_before_033", "oof": "oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "cv": 0.9700333620193362, "public_lb": 0.97043},
    {"key": "032_ovr_tabm", "exp_id": "exp_20260609_032_ovr_tabm_as_is", "family": "TabM", "role": "tabm_ovr_family_aux_material", "oof": "oof_exp_20260609_032_ovr_tabm_as_is_proba.npy", "pred": "pred_exp_20260609_032_ovr_tabm_as_is_proba.npy", "cv": 0.9610105651284856, "public_lb": 0.96106, "tuned_cv": 0.9686168281485955, "public_lb_biased": 0.96895},
    {"key": "033_blend_add032", "exp_id": "exp_20260609_033_blend_add032_tabm_check", "family": "Blend", "role": "previous_public_confirmed_best_backup", "oof": "oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "pred": "pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "cv": 0.9700400336552478, "public_lb": 0.97043},
    {"key": "034_gpu_logreg_default", "exp_id": "exp_20260609_034_gpu_logreg_stacker_own", "family": "GPU_LogisticRegression_Stacker", "role": "stacker_backup_default", "oof": "oof_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "pred": "pred_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "cv": 0.9700373069292101, "public_lb": 0.97022, "raw_cv": 0.9699389938897909, "public_lb_raw": 0.97027},
    {"key": "035_shared001_updated", "exp_id": "exp_20260610_035_shared001_updated_realmlp_pytorch_as_is", "family": "RealMLP", "role": "updated_shared001_realmlp_aux_material", "oof": "oof_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "pred": "pred_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "cv": 0.9687410900866934, "public_lb": 0.97012},
    {"key": "036_gpu_logreg_add035", "exp_id": "exp_20260610_036_gpu_logreg_stacker_add035_own", "family": "GPU_LogisticRegression_Stacker", "role": "improved_stacker_backup_add035", "oof": "oof_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "pred": "pred_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "cv": 0.9700674063284508, "public_lb": 0.97037},
    {"key": "036_blend_add035", "exp_id": "exp_20260610_036_blend_add035_shared001_check", "family": "Blend", "role": "cv_best_but_public_weak_add035_blend", "oof": "oof_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "pred": "pred_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "cv": 0.9700727843277738, "public_lb": 0.97023},
    {"key": "037_tabicl_v2", "exp_id": "exp_20260610_037_tabicl_v2_as_is", "family": "TabICL", "role": "weak_tabicl_family_material_optional_corr", "oof": "oof_exp_20260610_037_tabicl_v2_as_is_proba.npy", "pred": "pred_exp_20260610_037_tabicl_v2_as_is_proba.npy", "cv": 0.9580770153777599, "public_lb": 0.95920},
    {"key": "038_gpu_logreg_add037", "exp_id": "exp_20260610_038_gpu_logreg_stacker_add037_own", "family": "GPU_LogisticRegression_Stacker", "role": "previous_stacker_best_before_040", "oof": "oof_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy", "pred": "pred_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy", "cv": 0.9701353365798572, "public_lb": 0.97060},
    {"key": "039_cat_v3_seed2026_qbin_variant", "exp_id": "exp_20260611_039_cat_v3_seed2026_qbin_variant", "family": "CatBoost", "role": "catboost_v3_lower_correlation_variant_material", "oof": "oof_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy", "pred": "pred_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy", "cv": 0.9687053023092482, "public_lb": 0.96984},
    {"key": "040_gpu_logreg_add039", "exp_id": "exp_20260611_040_gpu_logreg_stacker_add039_own", "family": "GPU_LogisticRegression_Stacker", "role": "current_best_after_add039", "oof": "oof_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy", "pred": "pred_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy", "cv": 0.9702017877238539, "public_lb": 0.97069},
    {"key": "042_realmlp_v5_seed2026_foldvariant", "exp_id": "exp_20260611_042_realmlp_v5_seed2026_foldvariant", "family": "RealMLP", "role": "realmlp_v5_seed_fold_variant_material", "oof": "oof_exp_20260611_042_realmlp_v5_seed2026_foldvariant_proba.npy", "pred": "pred_exp_20260611_042_realmlp_v5_seed2026_foldvariant_proba.npy", "cv": 0.9689482884928097, "public_lb": 0.96943},
    {"key": "044_shared001_updated_seed2027_foldvariant", "exp_id": "exp_20260612_044_shared001_updated_seed2027_foldvariant", "family": "RealMLP", "role": "shared001_updated_seed_fold_variant_material", "oof": "oof_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy", "pred": "pred_exp_20260612_044_shared001_updated_seed2027_foldvariant_proba.npy", "cv": 0.9686035095582338, "public_lb": 0.97000},
    {"key": "045_gpu_logreg_add044", "exp_id": "exp_20260612_045_gpu_logreg_stacker_add044_own", "family": "GPU_LogisticRegression_Stacker", "role": "cv_best_but_public_weak_stacker_add044", "oof": "oof_exp_20260612_045_gpu_logreg_stacker_add044_own_proba.npy", "pred": "pred_exp_20260612_045_gpu_logreg_stacker_add044_own_proba.npy", "cv": 0.9702445493383917, "public_lb": 0.97040, "raw_cv": 0.9701651297689362, "tuned_cv": 0.9702445493383917},
    {"key": "047_xgb_multiclass_fe_objective_foldvariant", "exp_id": "exp_20260614_047_xgb_multiclass_fe_objective_foldvariant", "family": "XGBoost", "role": "failed_multiclass_xgb_variant_low_priority_diagnostic", "oof": "oof_exp_20260614_047_xgb_multiclass_fe_objective_foldvariant_proba.npy", "pred": "pred_exp_20260614_047_xgb_multiclass_fe_objective_foldvariant_proba.npy", "cv": 0.9596874103404002, "public_lb": 0.96027},
    {"key": "048_ovr_xgb_seed2027_fe_foldvariant", "exp_id": "exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant", "family": "XGBoost", "role": "improved_xgb_ovr_variant_material", "oof": "oof_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy", "pred": "pred_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy", "cv": 0.9636973807714805, "public_lb": 0.96444},
    {"key": "049_gpu_logreg_add048", "exp_id": "exp_20260614_049_gpu_logreg_stacker_add048_own", "family": "GPU_LogisticRegression_Stacker", "role": "add048_stacker_public_improved_diagnostic", "oof": "oof_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy", "pred": "pred_exp_20260614_049_gpu_logreg_stacker_add048_own_proba.npy", "cv": 0.9701958734042008, "public_lb": 0.97046, "raw_cv": 0.97011955078962, "tuned_cv": 0.9701958734042008},
]

TRACK_KEYS = {
    "049": "049_gpu_logreg_add048",
    "048": "048_ovr_xgb_seed2027_fe_foldvariant",
    "047": "047_xgb_multiclass_fe_objective_foldvariant",
    "045": "045_gpu_logreg_add044",
    "044": "044_shared001_updated_seed2027_foldvariant",
    "042": "042_realmlp_v5_seed2026_foldvariant",
    "040": "040_gpu_logreg_add039",
    "039": "039_cat_v3_seed2026_qbin_variant",
    "038": "038_gpu_logreg_add037",
    "037": "037_tabicl_v2",
    "036_gpu": "036_gpu_logreg_add035",
    "036_blend": "036_blend_add035",
    "035": "035_shared001_updated",
    "034": "034_gpu_logreg_default",
    "033": "033_blend_add032",
    "032": "032_ovr_tabm",
    "031": "031_blend_add030",
    "030": "030_ovr_xgb",
    "029": "029_blend_add028",
    "028": "028_cat_v3",
}

FOCUS_MODEL_KEYS = {
    "049": "049_gpu_logreg_add048",
    "048": "048_ovr_xgb_seed2027_fe_foldvariant",
    "047": "047_xgb_multiclass_fe_objective_foldvariant",
    "045": "045_gpu_logreg_add044",
    "040": "040_gpu_logreg_add039",
    "038": "038_gpu_logreg_add037",
    "033": "033_blend_add032",
    "031": "031_blend_add030",
    "030": "030_ovr_xgb",
    "044": "044_shared001_updated_seed2027_foldvariant",
    "042": "042_realmlp_v5_seed2026_foldvariant",
}

# 050 should answer:
# - Does 049 improve direct probability blending with 040/038/033/045?
# - Does 048 have direct probability-blend value beyond its use inside 049?
# - Does the weak 047 receive non-trivial safe-blend weight? If yes, consider a later GPU LogReg stacker add047 diagnostic.
# - Keep interpretation conservative: 047/048 are weak standalone models; useful only if safe-blend weights and Public LB confirm.
BLEND_SETS = {
    # Singles / current references
    "A_049_only": ["049_gpu_logreg_add048"],
    "B_045_only": ["045_gpu_logreg_add044"],
    "C_040_only": ["040_gpu_logreg_add039"],
    "D_038_only": ["038_gpu_logreg_add037"],
    "E_033_only": ["033_blend_add032"],
    "F_048_only": ["048_ovr_xgb_seed2027_fe_foldvariant"],
    "G_047_only": ["047_xgb_multiclass_fe_objective_foldvariant"],
    "H_044_only": ["044_shared001_updated_seed2027_foldvariant"],
    "I_042_only": ["042_realmlp_v5_seed2026_foldvariant"],
    "J_039_only": ["039_cat_v3_seed2026_qbin_variant"],
    "K_036_gpu_only": ["036_gpu_logreg_add035"],
    "L_036_blend_only": ["036_blend_add035"],
    "M_031_only": ["031_blend_add030"],

    # Main 049 checks.
    "N_049_040": ["049_gpu_logreg_add048", "040_gpu_logreg_add039"],
    "O_049_038": ["049_gpu_logreg_add048", "038_gpu_logreg_add037"],
    "P_049_033": ["049_gpu_logreg_add048", "033_blend_add032"],
    "Q_049_045": ["049_gpu_logreg_add048", "045_gpu_logreg_add044"],
    "R_049_040_038": ["049_gpu_logreg_add048", "040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "S_049_040_033": ["049_gpu_logreg_add048", "040_gpu_logreg_add039", "033_blend_add032"],
    "T_049_038_033": ["049_gpu_logreg_add048", "038_gpu_logreg_add037", "033_blend_add032"],
    "U_049_045_040": ["049_gpu_logreg_add048", "045_gpu_logreg_add044", "040_gpu_logreg_add039"],
    "V_049_045_038": ["049_gpu_logreg_add048", "045_gpu_logreg_add044", "038_gpu_logreg_add037"],
    "W_049_045_040_038": ["049_gpu_logreg_add048", "045_gpu_logreg_add044", "040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "X_049_045_040_038_033": ["049_gpu_logreg_add048", "045_gpu_logreg_add044", "040_gpu_logreg_add039", "038_gpu_logreg_add037", "033_blend_add032"],

    # 048 direct checks. 048 is better than 030/047 but still weak; expect low direct weight unless it provides useful diversity.
    "Y_048_040": ["048_ovr_xgb_seed2027_fe_foldvariant", "040_gpu_logreg_add039"],
    "Z01_048_038": ["048_ovr_xgb_seed2027_fe_foldvariant", "038_gpu_logreg_add037"],
    "Z02_048_033": ["048_ovr_xgb_seed2027_fe_foldvariant", "033_blend_add032"],
    "Z03_048_045": ["048_ovr_xgb_seed2027_fe_foldvariant", "045_gpu_logreg_add044"],
    "Z04_048_049": ["048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048"],
    "Z05_048_049_040_038": ["048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048", "040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "Z06_048_049_045_040_038_033": ["048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048", "045_gpu_logreg_add044", "040_gpu_logreg_add039", "038_gpu_logreg_add037", "033_blend_add032"],

    # 047 diagnostic checks. If 047 receives meaningful safe-blend weight, consider GPU LogReg stacker add047 next.
    "Z07_047_040": ["047_xgb_multiclass_fe_objective_foldvariant", "040_gpu_logreg_add039"],
    "Z08_047_038": ["047_xgb_multiclass_fe_objective_foldvariant", "038_gpu_logreg_add037"],
    "Z09_047_033": ["047_xgb_multiclass_fe_objective_foldvariant", "033_blend_add032"],
    "Z10_047_045": ["047_xgb_multiclass_fe_objective_foldvariant", "045_gpu_logreg_add044"],
    "Z11_047_049": ["047_xgb_multiclass_fe_objective_foldvariant", "049_gpu_logreg_add048"],
    "Z12_047_048_049": ["047_xgb_multiclass_fe_objective_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048"],
    "Z13_047_048_049_040_038": ["047_xgb_multiclass_fe_objective_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048", "040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "Z14_047_048_049_045_040_038_033": ["047_xgb_multiclass_fe_objective_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048", "045_gpu_logreg_add044", "040_gpu_logreg_add039", "038_gpu_logreg_add037", "033_blend_add032"],

    # Existing 046-style core sets for backward comparison.
    "Z15_045_040_038": ["045_gpu_logreg_add044", "040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "Z16_045_040_033_038": ["045_gpu_logreg_add044", "040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037"],
    "Z17_045_040_033_038_044_042": ["045_gpu_logreg_add044", "040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "044_shared001_updated_seed2027_foldvariant", "042_realmlp_v5_seed2026_foldvariant"],
    "Z18_041_old_best_like_plus_new049": ["040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "036_gpu_logreg_add035", "036_blend_add035", "039_cat_v3_seed2026_qbin_variant", "037_tabicl_v2", "044_shared001_updated_seed2027_foldvariant", "042_realmlp_v5_seed2026_foldvariant", "045_gpu_logreg_add044", "049_gpu_logreg_add048"],
    "Z19_040_033_038_036gpu_036blend_039_044_042_048_049": ["040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "036_gpu_logreg_add035", "036_blend_add035", "039_cat_v3_seed2026_qbin_variant", "044_shared001_updated_seed2027_foldvariant", "042_realmlp_v5_seed2026_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048"],

    # Component diagnostics. Interpret weights carefully because composite blends and components coexist.
    "Z20_component_diagnostics_xgb_and_core": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3", "039_cat_v3_seed2026_qbin_variant", "030_ovr_xgb", "032_ovr_tabm", "035_shared001_updated", "037_tabicl_v2", "047_xgb_multiclass_fe_objective_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048"],
    "Z21_full_all_loaded": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
        "029_blend_add028", "030_ovr_xgb", "031_blend_add030",
        "032_ovr_tabm", "033_blend_add032", "034_gpu_logreg_default",
        "035_shared001_updated", "036_gpu_logreg_add035", "036_blend_add035",
        "037_tabicl_v2", "038_gpu_logreg_add037", "039_cat_v3_seed2026_qbin_variant",
        "040_gpu_logreg_add039", "042_realmlp_v5_seed2026_foldvariant",
        "044_shared001_updated_seed2027_foldvariant", "045_gpu_logreg_add044",
        "047_xgb_multiclass_fe_objective_foldvariant", "048_ovr_xgb_seed2027_fe_foldvariant", "049_gpu_logreg_add048",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260614_050_blend_add047_048_049_check
OUTDIR: /kaggle/working/exp_20260614_050_blend_add047_048_049_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 28
n_blend_sets: 46


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 030_ovr_xgb: (577347, 3) / (247435, 3)
loaded 031_blend_add030: (577347, 3) / (247435, 3)
loaded 032_ovr_tabm: (577347, 3) / (247435, 3)
loaded 033_blend_add032: (577347, 3) / (247435, 3)
loaded 034_gpu_logreg_default: (577347, 3) / (247435, 3)
loaded 035_shared001_updated: (577347, 3) / (247435, 3)
loaded 036_gpu_logreg_add035: (577347, 3) / (247435, 3)
loaded 036_blend_add035: (577347, 3) / (247435, 3)
loaded 037_tabicl_v2: (577347, 3) / (247435, 3)
loaded 038_gpu_logreg_add037:

In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "public_lb_biased": spec.get("public_lb_biased", np.nan),
        "tuned_cv": spec.get("tuned_cv", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus tables for newly added / current reference models.
def focus_pairwise(key, filename):
    df = pairwise_df[(pairwise_df["left"] == key) | (pairwise_df["right"] == key)].copy()
    df = df.sort_values("pearson_flat_proba").reset_index(drop=True)
    display(df)
    df.to_csv(OUTDIR / filename, index=False)
    return df

focus_pairwise_tables = {}
for short_name, key in FOCUS_MODEL_KEYS.items():
    if key in model_keys:
        focus_pairwise_tables[short_name] = focus_pairwise(key, f"pairwise_oof_correlation_focus_{short_name}.csv")

# Backward-compatible variables for easy notebook inspection.
focus_040_df = focus_pairwise_tables.get("040", pd.DataFrame())
focus_039_df = focus_pairwise_tables.get("039", pd.DataFrame())
focus_038_df = focus_pairwise_tables.get("038", pd.DataFrame())
focus_037_df = focus_pairwise_tables.get("037", pd.DataFrame())
focus_033_df = focus_pairwise_tables.get("033", pd.DataFrame())


,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,tuned_cv,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,045_gpu_logreg_add044,exp_20260612_045_gpu_logreg_stacker_add044_own,GPU_LogisticRegression_Stacker,cv_best_but_public_weak_stacker_add044,0.970245,0.970400,NaN,0.970245,0.970245,0.0,0.960096,0.977148,0.973490
1,040_gpu_logreg_add039,exp_20260611_040_gpu_logreg_stacker_add039_own,GPU_LogisticRegression_Stacker,current_best_after_add039,0.970202,0.970690,NaN,NaN,0.970202,0.0,0.959500,0.976866,0.974240
2,049_gpu_logreg_add048,exp_20260614_049_gpu_logreg_stacker_add048_own,GPU_LogisticRegression_Stacker,add048_stacker_public_improved_diagnostic,0.970196,0.970460,NaN,0.970196,0.970196,0.0,0.960273,0.977199,0.973115
3,038_gpu_logreg_add037,exp_20260610_038_gpu_logreg_stacker_add037_own,GPU_LogisticRegression_Stacker,previous_stacker_best_before_040,0.970135,0.970600,NaN,NaN,0.970135,0.0,0.958120,0.977950,0.974336
4,036_blend_add035,exp_20260610_036_blend_add035_shared001_check,Blend,cv_best_but_public_weak_add035_blend,0.970073,0.970230,NaN,NaN,0.970073,0.0,0.960777,0.976943,0.972499
5,036_gpu_logreg_add035,exp_20260610_036_gpu_logreg_stacker_add035_own,GPU_LogisticRegression_Stacker,improved_stacker_backup_add035,0.970067,0.970370,NaN,NaN,0.970067,0.0,0.958313,0.978701,0.973188
6,033_blend_add032,exp_20260609_033_blend_add032_tabm_check,Blend,previous_public_confirmed_best_backup,0.970040,0.970430,NaN,NaN,0.970040,0.0,0.961775,0.975773,0.972571
7,034_gpu_logreg_default,exp_20260609_034_gpu_logreg_stacker_own,GPU_LogisticRegression_Stacker,stacker_backup_default,0.970037,0.970220,NaN,NaN,0.970037,0.0,0.959794,0.977771,0.972547
8,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_current_best_before_033,0.970033,0.970430,NaN,NaN,0.970033,0.0,0.961738,0.975790,0.972571
9,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_best_backup,0.970002,0.970036,NaN,NaN,0.970002,0.0,0.961315,0.975867,0.972825


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,047_xgb_multiclass_fe_objective_foldvariant,0.972636,0.953048,0.038251,0.049665,0.029753,0.020788,0.354564,0.028877,0.008965
1,030_ovr_xgb,037_tabicl_v2,0.974477,0.953569,0.037044,0.028955,0.049665,0.020975,0.363871,0.007980,0.028690
2,032_ovr_tabm,037_tabicl_v2,0.974850,0.953750,0.037186,0.029062,0.049665,0.020963,0.362909,0.008099,0.028702
3,037_tabicl_v2,048_ovr_xgb_seed2027_fe_foldvariant,0.978206,0.956579,0.033862,0.049665,0.029287,0.022716,0.403936,0.026949,0.006571
4,015_realmlp_seed42,037_tabicl_v2,0.979561,0.910951,0.031806,0.034388,0.049665,0.026350,0.456641,0.008038,0.023315
...,...,...,...,...,...,...,...,...,...,...,...
373,045_gpu_logreg_add044,049_gpu_logreg_add048,0.999960,0.998869,0.001136,0.034525,0.034452,0.033936,0.968464,0.000589,0.000516
374,040_gpu_logreg_add039,045_gpu_logreg_add044,0.999966,0.999174,0.001102,0.034865,0.034525,0.034158,0.969520,0.000707,0.000367
375,029_blend_add028,033_blend_add032,0.999993,0.999794,0.000438,0.034083,0.033838,0.033749,0.987632,0.000334,0.000088
376,029_blend_add028,031_blend_add030,0.999993,0.999801,0.000421,0.034083,0.033858,0.033768,0.988140,0.000315,0.000090


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,049_gpu_logreg_add048,0.984050,0.957335,0.027732,0.049665,0.034452,0.028345,0.508230,0.021320,0.006107
1,047_xgb_multiclass_fe_objective_foldvariant,049_gpu_logreg_add048,0.989001,0.949482,0.022016,0.029753,0.034452,0.021230,0.493995,0.008523,0.013223
2,030_ovr_xgb,049_gpu_logreg_add048,0.990255,0.947497,0.021169,0.028955,0.034452,0.021225,0.503162,0.007730,0.013228
3,032_ovr_tabm,049_gpu_logreg_add048,0.990478,0.955091,0.021084,0.029062,0.034452,0.021320,0.505275,0.007742,0.013132
4,048_ovr_xgb_seed2027_fe_foldvariant,049_gpu_logreg_add048,0.993048,0.951880,0.017257,0.029287,0.034452,0.023334,0.577503,0.005953,0.011118
5,018_xgb_shared003,049_gpu_logreg_add048,0.994308,0.968039,0.014147,0.033536,0.034452,0.026998,0.658624,0.006539,0.007455
6,015_realmlp_seed42,049_gpu_logreg_add048,0.996940,0.927431,0.010295,0.034388,0.034452,0.029391,0.745039,0.004997,0.005061
7,017_realmlp_bias,049_gpu_logreg_add048,0.997026,0.923623,0.010340,0.035542,0.034452,0.029940,0.747503,0.005601,0.004512
8,042_realmlp_v5_seed2026_foldvariant,049_gpu_logreg_add048,0.997536,0.853026,0.008517,0.035949,0.034452,0.031042,0.788682,0.004907,0.003410
9,044_shared001_updated_seed2027_foldvariant,049_gpu_logreg_add048,0.997714,0.893090,0.008679,0.035833,0.034452,0.030896,0.784398,0.004936,0.003556


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,048_ovr_xgb_seed2027_fe_foldvariant,0.978206,0.956579,0.033862,0.049665,0.029287,0.022716,0.403936,0.026949,0.006571
1,042_realmlp_v5_seed2026_foldvariant,048_ovr_xgb_seed2027_fe_foldvariant,0.990047,0.755354,0.019548,0.035949,0.029287,0.022971,0.543480,0.012978,0.006317
2,017_realmlp_bias,048_ovr_xgb_seed2027_fe_foldvariant,0.990246,0.892845,0.019574,0.035542,0.029287,0.022763,0.541113,0.012779,0.006525
3,026_realmlp_v5,048_ovr_xgb_seed2027_fe_foldvariant,0.990399,0.757243,0.019016,0.035388,0.029287,0.022948,0.549956,0.012440,0.006339
4,015_realmlp_seed42,048_ovr_xgb_seed2027_fe_foldvariant,0.991328,0.907443,0.018050,0.034388,0.029287,0.022953,0.563651,0.011435,0.006334
5,044_shared001_updated_seed2027_foldvariant,048_ovr_xgb_seed2027_fe_foldvariant,0.991437,0.882209,0.018930,0.035833,0.029287,0.023223,0.554302,0.012609,0.006064
6,035_shared001_updated,048_ovr_xgb_seed2027_fe_foldvariant,0.991656,0.889307,0.018188,0.034984,0.029287,0.023159,0.563322,0.011825,0.006128
7,039_cat_v3_seed2026_qbin_variant,048_ovr_xgb_seed2027_fe_foldvariant,0.991735,0.967212,0.018441,0.035587,0.029287,0.023350,0.562318,0.012237,0.005938
8,028_cat_v3,048_ovr_xgb_seed2027_fe_foldvariant,0.991857,0.968028,0.018032,0.035277,0.029287,0.023390,0.568063,0.011887,0.005898
9,038_gpu_logreg_add037,048_ovr_xgb_seed2027_fe_foldvariant,0.991878,0.939663,0.019054,0.035533,0.029287,0.022981,0.549263,0.012552,0.006306


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,047_xgb_multiclass_fe_objective_foldvariant,0.972636,0.953048,0.038251,0.049665,0.029753,0.020788,0.354564,0.028877,0.008965
1,042_realmlp_v5_seed2026_foldvariant,047_xgb_multiclass_fe_objective_foldvariant,0.986131,0.752138,0.023714,0.035949,0.029753,0.021140,0.474386,0.014809,0.008614
2,026_realmlp_v5,047_xgb_multiclass_fe_objective_foldvariant,0.986641,0.754358,0.023149,0.035388,0.029753,0.021148,0.480728,0.014239,0.008605
3,017_realmlp_bias,047_xgb_multiclass_fe_objective_foldvariant,0.986868,0.891112,0.023364,0.035542,0.029753,0.021129,0.478411,0.014412,0.008624
4,038_gpu_logreg_add037,047_xgb_multiclass_fe_objective_foldvariant,0.987637,0.937710,0.023721,0.035533,0.029753,0.020922,0.471578,0.014612,0.008832
5,044_shared001_updated_seed2027_foldvariant,047_xgb_multiclass_fe_objective_foldvariant,0.987750,0.880705,0.023059,0.035833,0.029753,0.021417,0.484883,0.014416,0.008336
6,039_cat_v3_seed2026_qbin_variant,047_xgb_multiclass_fe_objective_foldvariant,0.987780,0.965998,0.022865,0.035587,0.029753,0.021400,0.487012,0.014187,0.008354
7,036_gpu_logreg_add035,047_xgb_multiclass_fe_objective_foldvariant,0.987823,0.938261,0.023539,0.035419,0.029753,0.020954,0.473892,0.014464,0.008799
8,028_cat_v3,047_xgb_multiclass_fe_objective_foldvariant,0.987942,0.966078,0.022451,0.035277,0.029753,0.021433,0.491597,0.013844,0.008321
9,035_shared001_updated,047_xgb_multiclass_fe_objective_foldvariant,0.988141,0.887807,0.022281,0.034984,0.029753,0.021379,0.493069,0.013605,0.008375


...

,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,033_blend_add032,037_tabicl_v2,0.984870,0.865547,0.027770,0.033838,0.049665,0.028016,0.504916,0.005821,0.021649
1,033_blend_add032,047_xgb_multiclass_fe_objective_foldvariant,0.989490,0.824570,0.021062,0.033838,0.029753,0.021408,0.507514,0.012429,0.008345
2,030_ovr_xgb,033_blend_add032,0.990558,0.821940,0.020357,0.028955,0.033838,0.021344,0.514960,0.007611,0.012493
3,032_ovr_tabm,033_blend_add032,0.990801,0.830690,0.020230,0.029062,0.033838,0.021446,0.517361,0.007616,0.012391
4,033_blend_add032,048_ovr_xgb_seed2027_fe_foldvariant,0.993213,0.827104,0.016534,0.033838,0.029287,0.023405,0.589264,0.010432,0.005882
5,018_xgb_shared003,033_blend_add032,0.993883,0.858653,0.014542,0.033536,0.033838,0.026513,0.648849,0.007024,0.007325
6,015_realmlp_seed42,033_blend_add032,0.997205,0.868130,0.009731,0.034388,0.033838,0.029355,0.755191,0.005033,0.004483
7,017_realmlp_bias,033_blend_add032,0.997258,0.881385,0.009920,0.035542,0.033838,0.029828,0.754149,0.005714,0.004010
8,033_blend_add032,042_realmlp_v5_seed2026_foldvariant,0.997788,0.944717,0.008588,0.033838,0.035949,0.030692,0.785078,0.003145,0.005257
9,033_blend_add032,044_shared001_updated_seed2027_foldvariant,0.997856,0.858801,0.008497,0.033838,0.035833,0.030677,0.786701,0.003161,0.005156


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,031_blend_add030,037_tabicl_v2,0.984873,0.865479,0.027770,0.033858,0.049665,0.028026,0.505009,0.005832,0.021639
1,031_blend_add030,047_xgb_multiclass_fe_objective_foldvariant,0.989468,0.824466,0.021090,0.033858,0.029753,0.021405,0.507141,0.012454,0.008349
2,030_ovr_xgb,031_blend_add030,0.990536,0.821836,0.020385,0.028955,0.033858,0.021341,0.514576,0.007614,0.012518
3,031_blend_add030,032_ovr_tabm,0.990777,0.830573,0.020262,0.033858,0.029062,0.021441,0.516912,0.012417,0.007621
4,031_blend_add030,048_ovr_xgb_seed2027_fe_foldvariant,0.993196,0.827006,0.016562,0.033858,0.029287,0.023402,0.588817,0.010456,0.005886
5,018_xgb_shared003,031_blend_add030,0.993874,0.858577,0.014553,0.033536,0.033858,0.026518,0.648729,0.007018,0.007340
6,015_realmlp_seed42,031_blend_add030,0.997199,0.868061,0.009734,0.034388,0.033858,0.029364,0.755178,0.005025,0.004495
7,017_realmlp_bias,031_blend_add030,0.997253,0.881326,0.009913,0.035542,0.033858,0.029842,0.754368,0.005700,0.004017
8,031_blend_add030,042_realmlp_v5_seed2026_foldvariant,0.997786,0.944782,0.008584,0.033858,0.035949,0.030704,0.785214,0.003154,0.005245
9,031_blend_add030,044_shared001_updated_seed2027_foldvariant,0.997853,0.858744,0.008487,0.033858,0.035833,0.030692,0.786996,0.003166,0.005141


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,030_ovr_xgb,037_tabicl_v2,0.974477,0.953569,0.037044,0.028955,0.049665,0.020975,0.363871,0.007980,0.028690
1,030_ovr_xgb,042_realmlp_v5_seed2026_foldvariant,0.987105,0.749468,0.023140,0.028955,0.035949,0.021019,0.478944,0.007936,0.014930
2,026_realmlp_v5,030_ovr_xgb,0.987581,0.751435,0.022543,0.035388,0.028955,0.021039,0.485861,0.014348,0.007916
3,017_realmlp_bias,030_ovr_xgb,0.988127,0.889617,0.022371,0.035542,0.028955,0.021218,0.490255,0.014324,0.007737
4,030_ovr_xgb,044_shared001_updated_seed2027_foldvariant,0.988672,0.878785,0.022475,0.028955,0.035833,0.021294,0.489586,0.007661,0.014539
5,030_ovr_xgb,039_cat_v3_seed2026_qbin_variant,0.988783,0.964342,0.022205,0.028955,0.035587,0.021308,0.492849,0.007647,0.014279
6,028_cat_v3,030_ovr_xgb,0.988941,0.965161,0.021862,0.035277,0.028955,0.021330,0.497194,0.013947,0.007625
7,030_ovr_xgb,038_gpu_logreg_add037,0.989016,0.936473,0.022898,0.028955,0.035533,0.020903,0.479574,0.008052,0.014631
8,030_ovr_xgb,035_shared001_updated,0.989101,0.885761,0.021524,0.028955,0.034984,0.021342,0.501037,0.007612,0.013642
9,030_ovr_xgb,036_gpu_logreg_add035,0.989202,0.937636,0.022671,0.028955,0.035419,0.020958,0.482726,0.007997,0.014461


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,044_shared001_updated_seed2027_foldvariant,0.983156,0.882472,0.029218,0.049665,0.035833,0.028319,0.495274,0.021346,0.007514
1,044_shared001_updated_seed2027_foldvariant,047_xgb_multiclass_fe_objective_foldvariant,0.987750,0.880705,0.023059,0.035833,0.029753,0.021417,0.484883,0.014416,0.008336
2,030_ovr_xgb,044_shared001_updated_seed2027_foldvariant,0.988672,0.878785,0.022475,0.028955,0.035833,0.021294,0.489586,0.007661,0.014539
3,032_ovr_tabm,044_shared001_updated_seed2027_foldvariant,0.989198,0.883758,0.022169,0.029062,0.035833,0.021491,0.495151,0.007571,0.014341
4,018_xgb_shared003,044_shared001_updated_seed2027_foldvariant,0.991347,0.884525,0.017563,0.033536,0.035833,0.026031,0.600655,0.007505,0.009802
5,044_shared001_updated_seed2027_foldvariant,048_ovr_xgb_seed2027_fe_foldvariant,0.991437,0.882209,0.018930,0.035833,0.029287,0.023223,0.554302,0.012609,0.006064
6,039_cat_v3_seed2026_qbin_variant,044_shared001_updated_seed2027_foldvariant,0.995318,0.892239,0.012985,0.035587,0.035833,0.029345,0.697431,0.006242,0.006488
7,028_cat_v3,044_shared001_updated_seed2027_foldvariant,0.995321,0.892453,0.013127,0.035277,0.035833,0.029130,0.693898,0.006147,0.006703
8,015_realmlp_seed42,044_shared001_updated_seed2027_foldvariant,0.996538,0.876838,0.010829,0.034388,0.035833,0.029802,0.737316,0.004586,0.006031
9,017_realmlp_bias,044_shared001_updated_seed2027_foldvariant,0.996663,0.875192,0.010869,0.035542,0.035833,0.030353,0.739909,0.005189,0.005480


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,042_realmlp_v5_seed2026_foldvariant,0.982745,0.808008,0.029559,0.049665,0.035949,0.028224,0.491791,0.021441,0.007725
1,042_realmlp_v5_seed2026_foldvariant,047_xgb_multiclass_fe_objective_foldvariant,0.986131,0.752138,0.023714,0.035949,0.029753,0.021140,0.474386,0.014809,0.008614
2,030_ovr_xgb,042_realmlp_v5_seed2026_foldvariant,0.987105,0.749468,0.023140,0.028955,0.035949,0.021019,0.478944,0.007936,0.014930
3,032_ovr_tabm,042_realmlp_v5_seed2026_foldvariant,0.987614,0.759791,0.022775,0.029062,0.035949,0.021237,0.485142,0.007825,0.014712
4,042_realmlp_v5_seed2026_foldvariant,048_ovr_xgb_seed2027_fe_foldvariant,0.990047,0.755354,0.019548,0.035949,0.029287,0.022971,0.543480,0.012978,0.006317
5,018_xgb_shared003,042_realmlp_v5_seed2026_foldvariant,0.990835,0.795267,0.017691,0.033536,0.035949,0.026028,0.598924,0.007508,0.009921
6,028_cat_v3,042_realmlp_v5_seed2026_foldvariant,0.994809,0.806349,0.013633,0.035277,0.035949,0.028948,0.684706,0.006329,0.007001
7,039_cat_v3_seed2026_qbin_variant,042_realmlp_v5_seed2026_foldvariant,0.995016,0.804879,0.013191,0.035587,0.035949,0.029310,0.694122,0.006277,0.006639
8,015_realmlp_seed42,042_realmlp_v5_seed2026_foldvariant,0.996174,0.813865,0.011006,0.034388,0.035949,0.029778,0.734167,0.004611,0.006171
9,017_realmlp_bias,042_realmlp_v5_seed2026_foldvariant,0.996396,0.832858,0.010867,0.035542,0.035949,0.030420,0.740680,0.005122,0.005529


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    for short_name, target_key in TRACK_KEYS.items():
        row[f"includes_{short_name}"] = target_key in keys
        row[f"weight_{short_name}"] = _weight_of(keys, weights, target_key)
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only; do not use as final candidates.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(100))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_blend_tables = {}
for short_name, target_key in TRACK_KEYS.items():
    include_col = f"includes_{short_name}"
    if include_col in safe_blend_df.columns:
        df = safe_blend_df[safe_blend_df[include_col]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
        focus_blend_tables[short_name] = df
        display(df.head(80))
        df.to_csv(OUTDIR / f"safe_blend_diagnostics_focus_{short_name}.csv", index=False)

# Backward-compatible variables for easy notebook inspection.
focus_040_blend_df = focus_blend_tables.get("040", pd.DataFrame())
focus_039_blend_df = focus_blend_tables.get("039", pd.DataFrame())
focus_038_blend_df = focus_blend_tables.get("038", pd.DataFrame())
focus_037_blend_df = focus_blend_tables.get("037", pd.DataFrame())
focus_033_blend_df = focus_blend_tables.get("033", pd.DataFrame())



=== A_049_only ['049_gpu_logreg_add048'] ===

=== B_045_only ['045_gpu_logreg_add044'] ===

=== C_040_only ['040_gpu_logreg_add039'] ===

=== D_038_only ['038_gpu_logreg_add037'] ===

=== E_033_only ['033_blend_add032'] ===

=== F_048_only ['048_ovr_xgb_seed2027_fe_foldvariant'] ===

=== G_047_only ['047_xgb_multiclass_fe_objective_foldvariant'] ===

=== H_044_only ['044_shared001_updated_seed2027_foldvariant'] ===

=== I_042_only ['042_realmlp_v5_seed2026_foldvariant'] ===

=== J_039_only ['039_cat_v3_seed2026_qbin_variant'] ===

=== K_036_gpu_only ['036_gpu_logreg_add035'] ===

=== L_036_blend_only ['036_blend_add035'] ===

=== M_031_only ['031_blend_add030'] ===

=== N_049_040 ['049_gpu_logreg_add048', '040_gpu_logreg_add039'] ===

=== O_049_038 ['049_gpu_logreg_add048', '038_gpu_logreg_add037'] ===

=== P_049_033 ['049_gpu_logreg_add048', '033_blend_add032'] ===

=== Q_049_045 ['049_gpu_logreg_add048', '045_gpu_logreg_add044'] ===

=== R_049_040_038 ['049_gpu_logreg_add048', '040_

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.0,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_049_045_040_038_033,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",5,0.970264,"{""049_gpu_logreg_add048"": 0.19068397962763695,...",True,0.190684,False,NaN,...,0.973925,0.970264,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,W_049_045_040_038,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970264,"{""049_gpu_logreg_add048"": 0.18640369818787422,...",True,0.186404,False,NaN,...,0.973901,0.970264,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,W_049_045_040_038,nm_softmax_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970255,"{""049_gpu_logreg_add048"": 0.2388205117412317, ...",True,0.238821,False,NaN,...,0.973829,NaN,NaN,0.970255,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,Z15_045_040_038,hc_nonnegative_raw,"045_gpu_logreg_add044,040_gpu_logreg_add039,03...",3,0.970253,"{""045_gpu_logreg_add044"": 0.5144616346780971, ...",False,NaN,False,NaN,...,0.973937,0.970253,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,K_036_gpu_only,ridge_reg_raw_logit,036_gpu_logreg_add035,1,0.969234,None,False,NaN,False,NaN,...,0.964267,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
196,N_049_040,elasticnet_reg_raw_logit,"049_gpu_logreg_add048,040_gpu_logreg_add039",2,0.969229,None,True,NaN,False,NaN,...,0.964400,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
197,K_036_gpu_only,ridge_reg_raw_rank_logit,036_gpu_logreg_add035,1,0.969211,None,False,NaN,False,NaN,...,0.964073,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
198,K_036_gpu_only,ridgeclf_raw_rank_logit,036_gpu_logreg_add035,1,0.969211,None,False,NaN,False,NaN,...,0.964073,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_049_045_040_038_033,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",5,0.970264,"{""049_gpu_logreg_add048"": 0.19068397962763695,...",True,0.190684,False,NaN,...,0.973925,0.970264,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,W_049_045_040_038,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970264,"{""049_gpu_logreg_add048"": 0.18640369818787422,...",True,0.186404,False,NaN,...,0.973901,0.970264,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,W_049_045_040_038,nm_softmax_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970255,"{""049_gpu_logreg_add048"": 0.2388205117412317, ...",True,0.238821,False,NaN,...,0.973829,NaN,NaN,0.970255,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,Z15_045_040_038,hc_nonnegative_raw,"045_gpu_logreg_add044,040_gpu_logreg_add039,03...",3,0.970253,"{""045_gpu_logreg_add044"": 0.5144616346780971, ...",False,NaN,False,NaN,...,0.973937,0.970253,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Z02_048_033,nm_softmax_raw,"048_ovr_xgb_seed2027_fe_foldvariant,033_blend_...",2,0.970040,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 1.8166...",False,NaN,True,0.000018,...,0.972571,NaN,NaN,0.970040,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
96,Z02_048_033,hc_nonnegative_raw,"048_ovr_xgb_seed2027_fe_foldvariant,033_blend_...",2,0.970040,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 0.0, ""...",False,NaN,True,0.000000,...,0.972571,0.970040,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,Z18_041_old_best_like_plus_new049,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",11,0.970039,"{""040_gpu_logreg_add039"": 0.09090909090909091,...",True,0.090909,False,NaN,...,0.973817,NaN,NaN,0.970039,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
98,Z18_041_old_best_like_plus_new049,avg,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",11,0.970039,None,True,NaN,False,NaN,...,0.973817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.0,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_049_045_040_038_033,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",5,0.970264,"{""049_gpu_logreg_add048"": 0.19068397962763695,...",True,0.190684,False,NaN,...,0.973925,0.970264,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,W_049_045_040_038,hc_nonnegative_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970264,"{""049_gpu_logreg_add048"": 0.18640369818787422,...",True,0.186404,False,NaN,...,0.973901,0.970264,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,W_049_045_040_038,nm_softmax_raw,"049_gpu_logreg_add048,045_gpu_logreg_add044,04...",4,0.970255,"{""049_gpu_logreg_add048"": 0.2388205117412317, ...",True,0.238821,False,NaN,...,0.973829,NaN,NaN,0.970255,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,Z19_040_033_038_036gpu_036blend_039_044_042_04...,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",10,0.970253,"{""040_gpu_logreg_add039"": 0.08407327347212103,...",True,0.365774,True,0.0,...,0.973551,0.970253,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,Z13_047_048_049_040_038,avg,"047_xgb_multiclass_fe_objective_foldvariant,04...",5,0.969089,None,True,NaN,True,NaN,...,0.964992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,Z20_component_diagnostics_xgb_and_core,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.968865,None,True,NaN,True,NaN,...,0.964376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,Z04_048_049,avg,"048_ovr_xgb_seed2027_fe_foldvariant,049_gpu_lo...",2,0.968481,None,True,NaN,True,NaN,...,0.962393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67,Z11_047_049,avg,"047_xgb_multiclass_fe_objective_foldvariant,04...",2,0.967681,None,True,NaN,False,NaN,...,0.958827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z19_040_033_038_036gpu_036blend_039_044_042_04...,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",10,0.970253,"{""040_gpu_logreg_add039"": 0.08407327347212103,...",True,0.365774,True,0.000000,...,0.973551,0.970253,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z20_component_diagnostics_xgb_and_core,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.970250,"{""027_blend_add026"": 0.09266904710786944, ""026...",True,0.510185,True,0.000000,...,0.973079,0.970250,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z03_048_045,nm_softmax_raw,"048_ovr_xgb_seed2027_fe_foldvariant,045_gpu_lo...",2,0.970245,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 1.9751...",False,NaN,True,0.000020,...,0.973490,NaN,NaN,0.970245,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,Z03_048_045,hc_nonnegative_raw,"048_ovr_xgb_seed2027_fe_foldvariant,045_gpu_lo...",2,0.970245,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 0.0, ""...",False,NaN,True,0.000000,...,0.973490,0.970245,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Z14_047_048_049_045_040_038_033,hc_nonnegative_raw,"047_xgb_multiclass_fe_objective_foldvariant,04...",7,0.970220,"{""047_xgb_multiclass_fe_objective_foldvariant""...",True,0.233447,True,0.083548,...,0.972366,0.970220,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Z13_047_048_049_040_038,hc_nonnegative_raw,"047_xgb_multiclass_fe_objective_foldvariant,04...",5,0.970213,"{""047_xgb_multiclass_fe_objective_foldvariant""...",True,0.322251,True,0.000000,...,0.973478,0.970213,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Z06_048_049_045_040_038_033,hc_nonnegative_raw,"048_ovr_xgb_seed2027_fe_foldvariant,049_gpu_lo...",6,0.970212,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 0.0851...",True,0.230418,True,0.085110,...,0.972318,0.970212,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Z04_048_049,nm_softmax_raw,"048_ovr_xgb_seed2027_fe_foldvariant,049_gpu_lo...",2,0.970206,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 0.0013...",True,0.998695,True,0.001305,...,0.973115,NaN,NaN,0.970206,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,Y_048_040,nm_softmax_raw,"048_ovr_xgb_seed2027_fe_foldvariant,040_gpu_lo...",2,0.970204,"{""048_ovr_xgb_seed2027_fe_foldvariant"": 0.0002...",False,NaN,True,0.000278,...,0.974240,NaN,NaN,0.970204,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


...

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z20_component_diagnostics_xgb_and_core,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.970250,"{""027_blend_add026"": 0.09266904710786944, ""026...",True,0.510185,True,0.000000,...,0.973079,0.970250,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z21_full_all_loaded,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969814,"{""015_realmlp_seed42"": 0.027756865420468166, ""...",True,0.039721,True,0.041723,...,0.970057,NaN,NaN,0.969814,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z21_full_all_loaded,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969688,None,True,NaN,True,NaN,...,0.969368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z20_component_diagnostics_xgb_and_core,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.969527,"{""027_blend_add026"": 0.49613240081968846, ""026...",True,0.042539,True,0.058864,...,0.968232,NaN,NaN,0.969527,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z20_component_diagnostics_xgb_and_core,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.968865,None,True,NaN,True,NaN,...,0.964376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M_031_only,avg,031_blend_add030,1,0.970033,None,False,NaN,False,NaN,...,0.972571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M_031_only,nm_softmax_raw,031_blend_add030,1,0.970033,"{""031_blend_add030"": 1.0}",False,NaN,False,NaN,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,M_031_only,hc_nonnegative_raw,031_blend_add030,1,0.970033,"{""031_blend_add030"": 1.0}",False,NaN,False,NaN,...,0.972571,0.970033,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z21_full_all_loaded,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969814,"{""015_realmlp_seed42"": 0.027756865420468166, ""...",True,0.039721,True,0.041723,...,0.970057,NaN,NaN,0.969814,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z21_full_all_loaded,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969688,None,True,NaN,True,NaN,...,0.969368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z20_component_diagnostics_xgb_and_core,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.970250,"{""027_blend_add026"": 0.09266904710786944, ""026...",True,0.510185,True,0.000000,...,0.973079,0.970250,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z21_full_all_loaded,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969814,"{""015_realmlp_seed42"": 0.027756865420468166, ""...",True,0.039721,True,0.041723,...,0.970057,NaN,NaN,0.969814,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z21_full_all_loaded,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969688,None,True,NaN,True,NaN,...,0.969368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z20_component_diagnostics_xgb_and_core,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.969527,"{""027_blend_add026"": 0.49613240081968846, ""026...",True,0.042539,True,0.058864,...,0.968232,NaN,NaN,0.969527,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z20_component_diagnostics_xgb_and_core,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.968865,None,True,NaN,True,NaN,...,0.964376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z21_full_all_loaded,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969814,"{""015_realmlp_seed42"": 0.027756865420468166, ""...",True,0.039721,True,0.041723,...,0.970057,NaN,NaN,0.969814,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,Z21_full_all_loaded,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969688,None,True,NaN,True,NaN,...,0.969368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_049,weight_049,includes_048,weight_048,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z21_full_all_loaded,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.970277,"{""015_realmlp_seed42"": 0.025633413531209702, ""...",True,0.249955,True,0.000000,...,0.972910,0.970277,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z20_component_diagnostics_xgb_and_core,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.970250,"{""027_blend_add026"": 0.09266904710786944, ""026...",True,0.510185,True,0.000000,...,0.973079,0.970250,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z21_full_all_loaded,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969814,"{""015_realmlp_seed42"": 0.027756865420468166, ""...",True,0.039721,True,0.041723,...,0.970057,NaN,NaN,0.969814,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z21_full_all_loaded,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",28,0.969688,None,True,NaN,True,NaN,...,0.969368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z20_component_diagnostics_xgb_and_core,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.969527,"{""027_blend_add026"": 0.49613240081968846, ""026...",True,0.042539,True,0.058864,...,0.968232,NaN,NaN,0.969527,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z20_component_diagnostics_xgb_and_core,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",11,0.968865,None,True,NaN,True,NaN,...,0.964376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
for short_name, target_key in TRACK_KEYS.items():
    best_info[f"includes_{short_name}"] = target_key in best_keys
    best_info[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_record = {
    "set_name": best_set_name,
    "method": best_method,
    "cv": float(best_oof_score),
    "keys": ",".join(best_keys),
    "weights_json": best_row.get("weights_json"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}
for short_name, target_key in TRACK_KEYS.items():
    saved_submission_record[f"includes_{short_name}"] = target_key in best_keys
    saved_submission_record[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

saved_submission_df = pd.DataFrame([saved_submission_record])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "Z21_full_all_loaded",
  "method": "hc_nonnegative_raw",
  "keys": "015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,024_seed_ensemble_blend,026_realmlp_v5,027_blend_add026,028_cat_v3,029_blend_add028,030_ovr_xgb,031_blend_add030,032_ovr_tabm,033_blend_add032,034_gpu_logreg_default,035_shared001_updated,036_gpu_logreg_add035,036_blend_add035,037_tabicl_v2,038_gpu_logreg_add037,039_cat_v3_seed2026_qbin_variant,040_gpu_logreg_add039,042_realmlp_v5_seed2026_foldvariant,044_shared001_updated_seed2027_foldvariant,045_gpu_logreg_add044,047_xgb_multiclass_fe_objective_foldvariant,048_ovr_xgb_seed2027_fe_foldvariant,049_gpu_logreg_add048",
  "n_models": 28,
  "balanced_accuracy": 0.9702769901284838,
  "weights_json": "{\"015_realmlp_seed42\": 0.025633413531209702, \"017_realmlp_bias\": 0.0, \"018_xgb_shared003\": 0.0, \"019_blend_best\": 0.0, \"020_blend_bias\": 0.0, \"024_seed_ensemble_blend\": 0.0, \"026_realmlp_v5\": 0.0974

,set_name,method,cv,keys,weights_json,submission,oof_proba,pred_proba,includes_049,weight_049,...,includes_032,weight_032,includes_031,weight_031,includes_030,weight_030,includes_029,weight_029,includes_028,weight_028
0,Z21_full_all_loaded,hc_nonnegative_raw,0.970277,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...","{""015_realmlp_seed42"": 0.025633413531209702, ""...",submission_exp_20260614_050_blend_add047_048_0...,oof_exp_20260614_050_blend_add047_048_049_chec...,pred_exp_20260614_050_blend_add047_048_049_che...,True,0.249955,...,True,0.025633,True,0.025633,True,0.0,True,0.027625,True,0.035397


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "050_context": "blend add047/048/049 check based on 046",
    "049_gpu_logreg_add048_cv": 0.9701958734042008,
    "049_gpu_logreg_add048_public_lb": 0.97046,
    "048_ovr_xgb_variant_cv": 0.9636973807714805,
    "048_ovr_xgb_variant_public_lb": 0.96444,
    "047_xgb_multiclass_variant_cv": 0.9596874103404002,
    "047_xgb_multiclass_variant_public_lb": 0.96027,
    "046_blend_add042_044_045_cv": 0.9702533348410616,
    "046_blend_add042_044_045_public_lb": 0.97042,
    "045_gpu_logreg_add044_cv": 0.9702445493383917,
    "045_gpu_logreg_add044_public_lb": 0.97040,
    "044_shared001_updated_variant_cv": 0.9686035095582338,
    "044_shared001_updated_variant_public_lb": 0.97000,
    "042_realmlp_v5_variant_cv": 0.9689482884928097,
    "042_realmlp_v5_variant_public_lb": 0.96943,
    "041_blend_add039_gpu040_cv": 0.9702371689099357,
    "041_blend_add039_gpu040_public_lb": 0.97050,
    "040_gpu_logreg_add039_cv": 0.9702017877238539,
    "040_gpu_logreg_add039_public_lb": 0.97069,
    "038_gpu_logreg_add037_cv": 0.9701353365798572,
    "038_gpu_logreg_add037_public_lb": 0.97060,
    "033_cv": 0.9700400336552478,
    "033_public_lb": 0.97043,
    "036_gpu_logreg_add035_cv": 0.9700674063284508,
    "036_gpu_logreg_add035_public_lb": 0.97037,
    "036_blend_add035_cv": 0.9700727843277738,
    "036_blend_add035_public_lb": 0.97023,
    "031_blend_add030_cv": 0.9700333620193362,
    "031_blend_add030_public_lb": 0.97043,
}

best_focus_safe = {}
for short_name, df in focus_blend_tables.items():
    best_focus_safe[short_name] = None if len(df) == 0 else df.iloc[0].to_dict()

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_focus_safe": best_focus_safe,
    "main_questions": {
        "does_049_help_probability_blend": "Check focus_049 safe blends and weight_049 versus 040/038/045/046 references.",
        "does_048_help_direct_probability_blend": "Check focus_048 safe blends and weight_048. 048 may help directly or only through 049.",
        "does_047_help_despite_weak_standalone": "Check focus_047 safe blends and weight_047. If non-trivial, consider a later GPU LogReg stacker add047 diagnostic.",
        "does_033_remain_private_hedge": "Check blend sets containing 033 versus stacker-heavy 040/049 cluster.",
        "should_promote_050_best_blend": "Only if best safe blend improves CV and Public LB over 040. Ignore in-sample meta screening rows for submission decisions.",
    },
    "automatic_view": {
        "best_safe_includes_049": bool(best_info.get("includes_049")),
        "best_safe_weight_049": best_info.get("weight_049"),
        "best_safe_includes_048": bool(best_info.get("includes_048")),
        "best_safe_weight_048": best_info.get("weight_048"),
        "best_safe_includes_047": bool(best_info.get("includes_047")),
        "best_safe_weight_047": best_info.get("weight_047"),
        "best_safe_includes_045": bool(best_info.get("includes_045")),
        "best_safe_weight_045": best_info.get("weight_045"),
        "best_safe_includes_040": bool(best_info.get("includes_040")),
        "best_safe_weight_040": best_info.get("weight_040"),
        "best_safe_includes_038": bool(best_info.get("includes_038")),
        "best_safe_weight_038": best_info.get("weight_038"),
        "best_safe_includes_033": bool(best_info.get("includes_033")),
        "best_safe_weight_033": best_info.get("weight_033"),
        "best_safe_cv": float(best_info["cv_score"]),
        "040_reference_cv": reference_scores["040_gpu_logreg_add039_cv"],
        "040_reference_public_lb": reference_scores["040_gpu_logreg_add039_public_lb"],
        "049_reference_cv": reference_scores["049_gpu_logreg_add048_cv"],
        "049_reference_public_lb": reference_scores["049_gpu_logreg_add048_public_lb"],
        "047_stack_add_rule": (
            "If 047 gets non-trivial weight in a safe blend and the blend improves CV or Public LB, "
            "run a later GPU LogReg stacker add047 diagnostic. Otherwise keep 047 dropped."
        ),
        "promotion_rule": (
            "Promote 050 only if the best safe probability blend beats 040 on CV and confirms on Public LB. "
            "Do not promote in-sample LogReg/Ridge/ElasticNet diagnostic rows."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 047, 048, and 049 to the 046 candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus": {k: v.to_dict(orient="records") for k, v in focus_pairwise_tables.items()},
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus": {k: v.to_dict(orient="records") for k, v in focus_blend_tables.items()},
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
summary_path = OUTDIR / "blend_add047_048_049_summary.json"
save_json(summary, summary_path)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 047/048/049 to 046 pool",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-14",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 047 multiclass XGB diagnostic, "
            "048 improved OVR XGB variant, and 049 GPU LogReg add048 stacker, then decide whether a direct safe probability blend "
            "can improve over the current 040 best."
        ),
        "success_criteria": [
            "load existing OOF/pred npy files including 047/048/049 and current references 040/038/033/045",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations with focus tables for 049/048/047/040/038/033",
            "evaluate add047/add048/add049 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_prefix": "pairwise_oof_correlation_focus_*.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_prefix": "safe_blend_diagnostics_focus_*.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": summary_path.name,
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "040 is current best. 050 should only replace it if safe probability blend improves CV and Public LB. "
            "049 is the main new diagnostic target; 048 is a direct OVR-XGB material; 047 is low-priority and should only be escalated "
            "to GPU LogReg stacker add047 if it receives non-trivial safe-blend weight."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv and confirm 049/048/047 recomputed OOF CV",
            "Review safe_blend_diagnostics_focus_049.csv, focus_048.csv, focus_047.csv, focus_040.csv, focus_038.csv, focus_033.csv",
            "If 047 has meaningful safe-blend weight and the blend improves CV/LB, consider GPU LogReg stacker add047 next",
            "Submit only best safe blend, not in-sample meta rows",
            "Keep 040 as slot_1 unless 050 best safe blend confirms on Public LB",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add047_048_049_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260614_050_blend_add047_048_049_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_030.csv
 - pairwise_oof_correlation_focus_031.csv
 - pairwise_oof_correlation_focus_033.csv
 - pairwise_oof_correlation_focus_038.csv
 - pairwise_oof_correlation_focus_040.csv
 - pairwise_oof_correlation_focus_042.csv
 - pairwise_oof_correlation_focus_044.csv
 - pairwise_oof_correlation_focus_045.csv
 - pairwise_oof_correlation_focus_047.csv
 - pairwise_oof_correlation_focus_048.csv
 - pairwise_oof_correlation_focus_049.csv
 - pred_exp_20260614_050_blend_add047_048_049_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_028.csv
 - safe_blend_diagnostics_focus_029.csv
 - safe_blend_diagnostics_focus_030.csv
 - safe_blend_diagnostics_focus_031.csv
 - safe_blen